# From Zero to Claude: A Contract-Analysis Story

Welcome! In this notebook we're going to build up an understanding of how to use Claude to analyze real legal contracts — starting from the absolute basics and ending with two different ways to actually *talk* to Claude in production: the classic **Messages API** and the newer **Claude Agent SDK**.

Think of this notebook as a story with chapters. Each chapter builds on the one before it:

1. **Setup** — install the tools we need, including `pyboxen` so our outputs look clean and readable.
2. **How LLMs Behave** — tokens, context windows, sampling, non-determinism, and how to evaluate LLM output.
3. **Model & Reasoning** — picking the right Claude model and giving it room to think.
4. **Prompting** — zero-shot, one-shot, and few-shot prompting on real contracts.
5. **Talking to Claude: Messages API** — synchronous, streaming, async, and batch requests.
6. **Claude Agent SDK vs. the Messages API** — the same contract-analysis task, done two different ways, so you can see exactly how they differ.

By the end, you'll know not just *what* each concept means, but *when* you'd reach for it in a real legal-tech application.

Let's get started 🚀

### Prerequisites

Please download and review the following documents before proceeding:

1. **AWS1.pdf**
   [Download Link](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing)

2. **PROFRAC HOLDINGS, LLC Credit Agreement.pdf**
   [Download Link](https://drive.google.com/file/d/1UyOxeaEQsK5TFxXHI63PshoKmj0yjTmW/view?usp=sharing)

## Chapter 1 — Setup

First, let's install everything we'll need for this notebook:

1. **`anthropic`** — the official Python SDK for calling Claude through the Messages API.
2. **`PyMuPDF`** (`fitz`) — for reading text out of PDF contracts.
3. **`tiktoken`** — for showing how text gets broken down into tokens.
4. **`tqdm`** — progress bars, used internally by some of the packages above.
5. **`pyboxen`** — a small library that draws pretty boxes around text in the console. We'll use it throughout this notebook so Claude's responses are easy to read at a glance, instead of scrolling through walls of plain text.

We'll install the Claude Agent SDK later, in the chapter where we actually need it — no point installing things before their story starts.

In [1]:
!pip install -q -U anthropic
!pip install -q -U PyMuPDF tqdm tiktoken pyboxen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.0 MB/s eta 0:00:00


In [2]:
import anthropic
import fitz
import tiktoken
import asyncio

from pyboxen import boxen

### A little helper: `show()`

Every time we get a response back from Claude in this notebook, we're going to print it using a small helper called `show()`. Instead of a plain `print(...)`, it wraps the text in a titled, colored box using `pyboxen` — so it's immediately obvious *what* you're looking at (a response? an error? a comparison?) as you scroll through the notebook.

You'll see `show(title, text)` used everywhere from here on instead of a bare `print(...)`.

In [3]:
def show(title, text, color="cyan"):
    """Pretty-print Claude's output inside a titled, colored box."""
    print(boxen(str(text), title=title, color=color, padding=1))

# 🔑 Make Sure to Place Your Anthropic API Key

In [50]:
client = anthropic.Anthropic(
    api_key="Place Your Anthrophic API Key Here"
)

**⚠️ Note:** **In the cell below, you need to upload a file named `AWS1.pdf`.**
**You can download the file from the link below.**
[📥 Download AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing)

In [8]:
from google.colab import files
uploaded = files.upload()

Saving AWS1.pdf to AWS1.pdf


## Chapter 2 — How LLMs Behave

Imagine we are building an AI assistant that helps users understand contracts.

A user uploads a contract and asks:

> **"What is the termination notice period in this contract?"**

Before we build advanced features like RAG, agents, or tool use, let's first understand what happens when this request reaches an LLM.

We'll explore five important concepts, one at a time:

1. **Tokens** — How does an LLM process text?
2. **Context Window** — How much information can the model handle at once?
3. **Sampling** — How does the model choose what to generate?
4. **Non-determinism** — Why can the same prompt produce different responses?
5. **Evaluation** — How should we test LLM responses?

Let's start by loading a real contract and looking at how it's converted into **tokens**.

## Let's take a contract and try to analyse it without much instruction

First, we load the PDF and extract the text from it, then generate the token count of the text.

In [9]:
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text()

        pages.append({
            "page": page_number,
            "text": text
        })

    return pages

In [10]:
short_document = extract_text("/content/AWS1.pdf")

### Combine the Contract Text

The PDF text is stored page by page in `short_document`.

Here, we combine the text from all pages into a single string so that it can be passed to Claude for analysis.

`"\n"` adds a new line between each page's text to preserve readability.

In [11]:
contract_text = "\n".join(
    page["text"] for page in short_document
)

## 🧩 Tokens

A **token** is a small piece of text that an LLM uses to process and understand language.
LLMs don't read text exactly like humans do. Before processing text, they break it into smaller pieces called tokens.

From the code below, we'll see how a real piece of our contract is converted into tokens. We will also compare the original text's number of characters with the number of tokens created by the tokenizer.

We'll use OpenAI's `cl100k_base` tokenizer just to *illustrate* the concept of tokenization — the same idea applies to Claude's own tokenizer.

#### What are we going to do?

We'll:
1. Take the text from the first page of our contract.
2. Convert the text into tokens.
3. Count the original **characters**.
4. Count the resulting **tokens**.
5. Compare both numbers.

In [13]:
encoding = tiktoken.get_encoding("cl100k_base")

sample = short_document[0]["text"]
tokens = encoding.encode(sample)

preview = sample[:300].replace(chr(10), " ")

token_pieces = [encoding.decode([t]) for t in tokens[:25]]
split_preview = "|".join(token_pieces)

show("📄 Original Text (first 300 chars)", preview + "...", color="yellow")

# 🔢 The actual tokenizer output: a list of integer token IDs
print("Token IDs (first 25):")
print(tokens[:25])

show("🔎 First 25 Tokens (split visualized)", split_preview, color="magenta")

show("🧩 Tokenization Summary", f"Characters: {len(sample)}\nTokens: {len(tokens)}", color="cyan")

╭─ 📄 Original Text (first 300 chars) ────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│    AWS Customer Agreement Last Updated: April 20, 2023 See What's Changed This AWS Customer Agreement (this     │
│    “Agreement”) contains the terms and conditions that govern your access to and use of the Services (as        │
│    deﬁned below)  and is an agreement between the applicable AWS Contracting Party speciﬁed in S...             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Token IDs (first 25):
[37236, 12557, 23314, 198, 5966, 16459, 25, 5936, 220, 508, 11, 220, 2366, 18, 198, 10031, 3639, 596, 47394, 198, 2028, 24124, 12557, 23314, 320]


╭─ 🔎 First 25 Tokens (split visualized) ─────╮                                                                    
│                                             │                                                                    
│   AWS| Customer| Agreement|                 │                                                                    
│   |Last| Updated|:| April| |20|,| |202|3|   │                                                                    
│   |See| What|'s| Changed|                   │                                                                    
│   |This| AWS| Customer| Agreement| (        │                                                                    
│                                             │                                                                    
╰─────────────────────────────────────────────╯                                                                    



╭─ 🧩 Tokenization Summary ─╮                                                                                      
│                           │                                                                                      
│     Characters: 3852      │                                                                                      
│     Tokens: 923           │                                                                                      
│                           │                                                                                      
╰───────────────────────────╯                                                                                      



**⚠️ Note:** **In the cell below, you need to upload a file named `PROFRAC HOLDINGS, LLC credit agreement.pdf`.**
**You can download the file from the link below.**
[📥 Download PROFRAC HOLDINGS, LLC credit agreement.pdf](https://drive.google.com/file/d/1UyOxeaEQsK5TFxXHI63PshoKmj0yjTmW/view?usp=sharing)

In [14]:
from google.colab import files
uploaded = files.upload()

Saving PROFRAC HOLDINGS, LLC credit agreement.pdf to PROFRAC HOLDINGS, LLC credit agreement.pdf


In [15]:
long_document = extract_text("/content/PROFRAC HOLDINGS, LLC credit agreement.pdf")

## Context Window

Now that we've seen tokens up close, let's zoom out.

A **context window** is the maximum amount of tokenized information an LLM can handle in a single request.

Think of it like the model's **working memory**. Everything we send to the model needs to fit inside this space — not just the contract.

A model can only process a limited amount of information in a single request.

Let's test this with our two contracts: a short 12-page one and a long 216-page one.

In [16]:
import base64

with open("/content/AWS1.pdf", "rb") as f:
    pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")

This is simply saying:
Open the PDF, read its contents, convert it into Base64, and store it in `pdf_data`.

```
Open the PDF → read the PDF → convert it into Base64 → store it in pdf_data.
```

Now `pdf_data` contains our PDF, so we send it to Claude with an instruction to analyze and summarize the contract.
We will run this request on both contracts to see what happens when the document becomes too large.

In [17]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_data
                    }
                },
                {
                    "type": "text",
                    "text": "Analyze this contract and summarize the key terms."
                }
            ]
        }
    ]
)

show("📄 12-page contract — Claude's summary", response.content[0].text, color="green")

╭─ 📄 12-page contract — Claude's summary ─────────────────────────────────────────────────────╮                   
│                                                                                              │                   
│   # AWS Customer Agreement - Key Terms Summary                                               │                   
│                                                                                              │                   
│   ## Contract Basics                                                                         │                   
│   - **Parties:** AWS (Amazon Web Services) and XYZ Software Solutions                        │                   
│   - **Effective Date:** April 1, 2023                                                        │                   
│   - **Duration:** 12 months, ending March 31, 2024                                           │                   
│   - **Contract Value:** USD 35,000 (paid annually before start of work)

In [18]:
with open("/content/PROFRAC HOLDINGS, LLC credit agreement.pdf", "rb") as f:
    pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")

In [19]:
try:

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "document",
                        "source": {
                            "type": "base64",
                            "media_type": "application/pdf",
                            "data": pdf_data
                        }
                    },
                    {
                        "type": "text",
                        "text": "Analyze this contract and summarize the key terms."
                    }
                ]
            }
        ]
    )

    show("📚 216-page contract — Claude's summary", response.content[0].text, color="green")

except Exception as e:

    show(
        "❌ 216-page contract — Failed",
        "Reason: The PDF is too large for a single request.\n"
        "Claude allows a maximum of 100 PDF pages for this request.",
        color="red"
    )

╭─ ❌ 216-page contract — Failed ────────────────────────────────╮                                                 
│                                                                │                                                 
│   Reason: The PDF is too large for a single request.           │                                                 
│   Claude allows a maximum of 100 PDF pages for this request.   │                                                 
│                                                                │                                                 
╰────────────────────────────────────────────────────────────────╯                                                 



**12-page contract**
→ Send the complete PDF to Claude
→ ✅ Successfully processed

**216-page contract**
→ Send the complete PDF to Claude
→ ❌ Request rejected because the PDF exceeds the allowed page limit

This shows an important limitation of processing large documents in a single request — and it's exactly the kind of constraint that motivates techniques like chunking and RAG, which are outside the scope of this notebook but worth keeping in mind.

## 🎲 Sampling

Now let's look at *how* Claude actually generates a response, one piece at a time.

When an LLM generates a response, it predicts the **next token** based on the tokens that came before it.

For each next token, the model assigns different probabilities.

```text
Possible next tokens

"30"       → 60%
"thirty"   → 20%
"90"       → 10%
"one"      →  5%
"other"    →  5%
```

The model then **samples** from these possibilities.

The `temperature` parameter controls how much randomness is introduced during sampling.

```python
extra_body={"temperature": 0-1}
```

* Low temperature → more predictable and consistent outputs
* High temperature → more variation and less predictable outputs

Let's ask the **same contract question 3 times** with two different temperatures and compare the results.

In [31]:
question = """
Write a short paragraph summarizing this contract's termination
provisions in your own words, as if explaining it to a colleague.
"""

### 🌡️ Temperature = 0

With a lower temperature, the model is more likely to choose the **most probable tokens**, so the responses tend to be more consistent.

In [32]:
for i in range(1):

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        system="You are a contract analysis assistant.",
        # 🌡️ Temperature controls response variation
        extra_body={"temperature": 0},
        messages=[
            {
                "role": "user",
                "content": question + "\n\n" + contract_text
            }
        ]
    )

    show(f"🌡️ Temperature 0 — Response {i + 1}", response.content[0].text, color="blue")

╭─ 🌡️ Temperature 0 — Response 1 ──────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   # Summary of Termination Provisions                                                                           │
│                                                                                                                 │
│   The termination section outlines how either party can end this AWS agreement. You can terminate anytime for   │
│   any reason by closing your account, while AWS can terminate with 30 days' notice. Either party can also       │
│   terminate for cause if the other materially breaches the agreement and doesn't fix it within 30 days of       │
│   notice. AWS can terminate immediately if there's a security risk, fraud, payment issues, or if a              │
│   third-party technology partner relationship ends. Importantly, the 

### 🔥 Temperature = 1

With a higher temperature, the model allows **more variation** when choosing tokens, so the same question can produce different responses.

In [34]:
for i in range(1):

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        system="You are a contract analysis assistant.",
        # 🌡️ Temperature controls response variation
        extra_body={"temperature": 1},
        messages=[
            {
                "role": "user",
                "content": question + "\n\n" + contract_text
            }
        ]
    )

    show(f"🔥 Temperature 1 — Response {i + 1}", response.content[0].text, color="Blue")

╭─ 🔥 Temperature 1 — Response 1 ─────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   # Summary of Termination Provisions                                                                           │
│                                                                                                                 │
│   This contract runs for a fixed 12-month term (April 1, 2023 to March 31, 2024) and does not automatically     │
│   renew. Either party can terminate for convenience—you can end it anytime by closing your account, and AWS     │
│   can end it with 30 days' notice. For termination for cause, either party must provide written notice of a     │
│   material breach and give the other 30 days to fix it; if not fixed, termination takes effect. AWS can also    │
│   terminate immediately if there's a security risk, payment issues, sus

### 🔍 What did we observe?

* **Higher temperature** → more randomness and potentially more varied responses
* **Lower temperature** → more predictable and consistent responses
* The underlying model is still performing the same task; the difference is in how tokens are selected during generation.

💡 **Key takeaway:** LLM outputs are probabilistic. Therefore, the same prompt does not always guarantee the exact same wording.

### 🔄 From Sampling to Non-Determinism

Look at the three responses we generated using the **same question and the same contract**.

At a higher temperature, the model may make different token choices, which can lead to different responses.

This is called **non-determinism**:

> **The same input does not necessarily produce the exact same output every time.**

For example:

```text
Run 1 → "The termination notice period is 30 days."
Run 2 → "The contract requires 30 days' notice for termination."
Run 3 → "Either party must provide 30 days notice before termination."
```

The wording is different, but the **meaning is the same**.

💡 This creates an important challenge when testing LLM applications:
**We shouldn't always test whether the output is exactly the same — we should test whether the output is correct.**

### 🧪 How Should We Test LLM Applications?

Traditional software often uses **exact-match testing**:

```text
Expected output == Actual output
```

For LLM applications, this can be too strict because multiple responses can be correct.

Instead, we can evaluate:

* **Correctness** — Did the model identify the right term?
* **Relevance** — Did it answer the question?
* **Completeness** — Did it include the important information?
* **Grounding** — Is the answer supported by the contract?

So instead of asking:

> "Did the model produce exactly this sentence?"

we ask:

> **"Did the model produce a correct answer?"**

This is one of the key differences between testing traditional software and testing LLM applications.

## Chapter 3 — Model & Reasoning

So far, we've seen how an LLM processes tokens, works within a context window, and generates probabilistic responses.

Now let's ask two important questions:

1. **Which model should we use?**
2. **When does a task require deeper reasoning?**

Not every task needs the same level of intelligence.

For example:

* Extracting a contract date → relatively simple
* Summarizing a clause → moderate
* Determining whether a termination clause creates a business risk → more complex

Let's see how **model selection** and **reasoning effort** affect our application.

## 🎯 Choosing the Right Model

Different models offer different trade-offs between:

* **Capability** — How well the model handles complex tasks
* **Latency** — How quickly it responds
* **Cost** — How expensive each request is

There is no single "best" model for every task.

Instead, we choose a model based on what our application actually needs.

### ⚖️ Model Trade-offs

| Model      | Capability | Latency   | Cost      |
| ---------- | ---------- | --------- | --------- |
| **Haiku**  | Medium     | 🟢 Low    | 🟢 Low    |
| **Sonnet** | High       | 🟡 Medium | 🟡 Medium |
| **Opus**   | Very High  | 🔴 Higher | 🔴 Higher |

💡 **Key idea:** Choose the model based on the task — not every task needs the most capable model.

## 🧠 Reasoning

**Reasoning = giving the model more time/tokens to think through complex tasks.**

* **Low effort** → faster + cheaper
* **High effort** → deeper reasoning + potentially higher cost/latency
* **✨ Adaptive thinking** → the model decides when more reasoning is useful

### ✨ Adaptive Thinking

With **adaptive thinking**, we don't manually decide how much reasoning the model should use.

Instead, Claude dynamically decides **when additional reasoning is needed** based on the complexity of the task.

In our example, adaptive thinking is enabled here:

```python
thinking={"type": "adaptive"}
```

We also set:

```python
output_config={"effort": "high"}
```

This tells Claude to use a **high reasoning effort**, while adaptive thinking allows the model to determine how much reasoning is actually useful.

👀 **Now let's see this in action.**

Run the prompt below and inspect the response.

You will see that Claude can return separate **thinking** and **text** blocks:

* 🧠 **Thinking block** → reasoning content returned by the API
* 💬 **Text block** → Claude's final answer

This lets us observe how **adaptive thinking** is being used for a complex contract-analysis task.

In [35]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    # ✨ Adaptive Thinking: Claude decides when reasoning is useful
    thinking={"type": "adaptive"},
    # Ask Claude to use high reasoning effort
    output_config={"effort": "high"},
    messages=[
        {
            "role": "user",
            "content": f"""
You are a contract risk analyst.

Analyze the following AWS Customer Agreement and determine:

1. What termination rights does each party have?
2. What notice periods apply?
3. What happens to the customer's access and content after termination?
4. Does this create any significant risk for the customer?
5. Give a concise recommendation to the contract reviewer.

Support your conclusion with the relevant section numbers.

Contract:

{contract_text}
"""
        }
    ]
)

# Display Claude's reasoning and final answer separately, each in its own box
for block in response.content:

    if block.type == "thinking":
        show("🧠 Claude's Reasoning", block.thinking, color="purple")

    elif block.type == "text":
        show("💬 Final Answer", block.text, color="green")

╭─ 🧠 Claude's Reasoning ─────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   Let me analyze the AWS Customer Agreement provided for XYZ Software Solutions, focusing on the termination    │
│   rights, notice periods, post-termination access, risks, and recommendations.                                  │
│                                                                                                                 │
│   Let me go through each question systematically:                                                               │
│                                                                                                                 │
│   **1. What termination rights does each party have?**                                                          │
│                                                                        

## Chapter 4 — Prompting: Zero-Shot, One-Shot & Few-Shot

**Prompting is the process of giving an LLM instructions, context, and examples to guide it toward the desired output.**

In this chapter, we will use a **contract PDF** and ask Claude to extract important contract terms.

We will progressively add examples to the prompt and observe how the output changes.

### Zero-Shot
No examples are provided.

**Task + Context**

Use when the task is simple and the model can infer the expected behavior.

### One-Shot
One example is provided.

**Task + Example + Context**

Use when we want to show the model the expected output format or style.

### Few-Shot
Multiple examples are provided.

**Task + Examples + Context**

Use when the task has different cases or edge cases that are difficult to describe using instructions alone.

> More examples do not always mean better results.
> The goal is to provide **just enough guidance** while keeping the prompt small.

We will compare the outputs and see whether additional examples improve the result.

### 🎯 Zero-Shot Prompt

In [36]:
prompt = f"""
Extract the key terms from this contract.

Return:
- Parties
- Effective Date
- Expiration Date
- Contract Value
- Payment Terms
- Termination Terms
- Governing Law

Contract:
{contract_text}
"""

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

show("🎯 Zero-Shot — Extracted Terms", response.content[0].text, color="cyan")

╭─ 🎯 Zero-Shot — Extracted Terms ────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   # AWS Customer Agreement - Key Terms Extract                                                                  │
│                                                                                                                 │
│   ## **Parties**                                                                                                │
│   - **Provider:** Amazon Web Services (AWS) / Applicable AWS Contracting Party (varies by Account Country -     │
│   see Section 12)                                                                                               │
│   - **Customer:** XYZ Software solutions (and/or the individual/entity representing them)                       │
│                                                                        

### 🥇 One-Shot Prompt

In [38]:
prompt = f"""
Extract the key terms from the contract.

Follow the output format shown in this example.

Example:

Contract:
"ABC Corp and XYZ Ltd entered into an agreement
effective January 1, 2025 for $50,000."

Output:
{{
    "parties": ["ABC Corp", "XYZ Ltd"],
    "effective_date": "2025-01-01",
    "contract_value": 50000
}}

Now extract the key terms from this contract:

{contract_text}
"""

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

show("🥇 One-Shot — Extracted Terms", response.content[0].text, color="cyan")

╭─ 🥇 One-Shot — Extracted Terms ───────────────────────────────────────────────────────────────────────╮          
│                                                                                                       │          
│   ```json                                                                                             │          
│   {                                                                                                   │          
│       "parties": ["Amazon Web Services (AWS)", "XYZ Software solutions"],                             │          
│       "effective_date": "2023-04-01",                                                                 │          
│       "contract_value": 35000,                                                                        │          
│       "contract_value_currency": "USD",                                                               │          
│       "payment_frequency": "annually",                                 

### 🏅 Few-Shot Prompt

In [39]:
prompt = f"""
Extract key terms from the contract.

Use these examples to understand how different
types of information should be represented.

Example 1 — Text:

Contract:
"The agreement is between ABC Corp and XYZ Ltd."

Output:
{{
    "parties": ["ABC Corp", "XYZ Ltd"]
}}


Example 2 — Number:

Contract:
"The total contract value is USD 50,000."

Output:
{{
    "contract_value": 50000
}}


Example 3 — Date:

Contract:
"The agreement becomes effective on January 1, 2025."

Output:
{{
    "effective_date": "2025-01-01"
}}


Example 4 — Missing value:

Contract:
"The agreement becomes effective on January 1, 2025."

Output:
{{
    "effective_date": "2025-01-01",
    "expiration_date": null
}}


Now apply these examples to the following contract:

{contract_text}
"""

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

show("🏅 Few-Shot — Extracted Terms", response.content[0].text, color="cyan")

╭─ 🏅 Few-Shot — Extracted Terms ─────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   ```json                                                                                                       │
│   {                                                                                                             │
│     "parties": [                                                                                                │
│       "Amazon Web Services (AWS)",                                                                              │
│       "XYZ Software Solutions"                                                                                  │
│     ],                                                                                                          │
│     "contract_type": "AWS Customer Agreement",                         

## What did we observe?

| Prompt | Examples | Purpose |
|---|---:|---|
| Zero-shot | 0 | Let the model infer the task |
| One-shot | 1 | Demonstrate the expected format |
| Few-shot | 2–4 | Demonstrate different cases and edge cases |

The important idea is **not**:

> "Add more examples whenever the output is bad."

Instead, ask:

> "What is the smallest amount of guidance that gives us the quality we need?"

If adding more examples makes the prompt huge, we should also consider changing the **model, context, reasoning effort, or output constraints** rather than creating a 150-page prompt.

## Chapter 5 — Talking to Claude: The Messages API

So far, every single call we've made has followed the same shape: send a request, wait, get a response back — much like a classic "completion" call. That's the **Messages API**, and it's the foundation everything else in this notebook is built on.

But *how* we wait for that response can change depending on what our application needs:

- **Synchronous:** Send a request and wait until the complete response is returned.
- **Streaming:** Receive the response progressively as Claude generates it.
- **Async:** Start requests without blocking the application while waiting for results.
- **Batching:** Submit many requests together, which is useful for evaluations and large-scale processing.

In this chapter, we will use a contract-analysis task and see how each approach works — and then, in the next (and final) chapter, we'll step back and see how the **Claude Agent SDK** offers a different way to work with Claude entirely.

Normal:

```
Request A
   ↓
wait
   ↓
Response A

Request B
   ↓
wait
   ↓
Response B
```

v/s

```
Async:

Request A ────────────────┐
Request B ──────────┐     │
Request C ───────┐  │     │
                 ↓  ↓     ↓
              Claude API
                 ↓
             Responses
```

### 1️⃣ Synchronous request

In [40]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    # Send the contract and instructions to Claude
    messages=[
        {
            "role": "user",
            "content": f"""
Analyze this contract and identify:

1. Contract parties
2. Effective date
3. Expiration date
4. Contract value
5. Termination terms

Contract:
{contract_text}
"""
        }
    ]
)

# This line runs only after Claude has returned the response
show("1️⃣ Synchronous — Full Response", response.content[0].text, color="cyan")

╭─ 1️⃣ Synchronous — Full Response ─────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   # AWS Customer Agreement Analysis                                                                             │
│                                                                                                                 │
│   ## 1. Contract Parties                                                                                        │
│                                                                                                                 │
│   | Party | Details |                                                                                           │
│   |-------|---------|                                                                                           │
│   | **Service Provider** | Amazon Web Services (AWS) / applicable AW

### What happened?

The request blocks until Claude finishes generating the complete response.

**Request → Wait → Complete Response**

### 2️⃣ Streaming response

In the previous example, we made a synchronous API call and waited for Claude to return the complete response.

With streaming, Claude sends the response piece by piece as it is generated.

Instead of waiting for the entire answer, we can display each piece immediately. We'll print the raw stream first (so you can watch it arrive token by token), then show the finished result in our usual box once it's done.

### 🔄 How Streaming Works

When we use:
```python
client.messages.stream(...)
```

Claude starts generating the response and sends the generated text in small chunks.

Our code then reads those chunks using:

```python
for text in stream.text_stream:
    ...
```

In [ ]:
streamed_text = ""

with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": f"""
Analyze this contract and identify the key termination
rights and notice periods.

Contract:
{contract_text}
"""
        }
    ]
) as stream:

    for text in stream.text_stream:
        print(text, end="", flush=True)
        streamed_text += text

print()  # tidy newline once streaming finishes
show("2️⃣ Streaming — Final Assembled Response", streamed_text, color="cyan")

### What changed?

With streaming, we don't wait for the entire response.

Claude sends generated text incrementally:

**Request → Token/Chunk → Token/Chunk → Token/Chunk → ... → Complete**

This is useful for chat interfaces where we want the user to see the response immediately.

### 3️⃣ Async / non-blocking request

## Asynchronous API Call

We can also call Claude **asynchronously** using `AsyncAnthropic`.

Unlike a synchronous call, an asynchronous call allows Python to work with other tasks while waiting for Claude's response.

### 🔑 Authentication

Because we are using Anthropic's API, we need an **Anthropic API key**.

Make sure you replace the placeholder below with your Anthropic API key.

> ⚠️ Do not use an OpenAI API key here. `AsyncAnthropic` requires an Anthropic API key.

### 🔄 Execution Flow

```text
Start async function
       ↓
Send request to Claude
       ↓
      await ⏳
       ↓
Receive response
       ↓
Return result
```

The important part is:

```python
response = await async_client.messages.create(...)
```

The `await` tells Python to **wait for this asynchronous operation to complete** before using the response.

We use `async def` to define the asynchronous function and `await` when making the API call.

In [41]:
import asyncio
from anthropic import AsyncAnthropic

async_client = AsyncAnthropic(
    api_key="Place Your Anthropic API Key Here"
)

async def analyze_contract():
    response = await async_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": f"""
Analyze this contract and summarize the termination
rights and associated risks.

Contract:
{contract_text}
"""
            }
        ]
    )

    return response.content[0].text


result = await analyze_contract()

show("3️⃣ Async — Response", result, color="cyan")

AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011Cf1LioM2BZx17vyjeZeji'}

### Why async?

Async allows our application to start an API call without blocking other work.

This becomes especially useful when we have multiple independent tasks that can run at the same time.

### 4️⃣ Multiple async calls

In [ ]:
async def ask_claude(prompt):
    response = await async_client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.content[0].text


prompts = [
    "Identify the contract parties.",
    "Identify the contract value.",
    "Identify the effective date.",
    "Identify the termination terms.",
    "Identify the governing law."
]

results = await asyncio.gather(
    *(ask_claude(prompt + f"\n\nContract:\n{contract_text}")
      for prompt in prompts)
)

for i, result in enumerate(results, 1):
    show(f"4️⃣ Async Batch — Result {i}", result, color="cyan")

## 5️⃣ Batching Contract Evals

When evaluating a model across multiple contracts, we may want to ask the **same set of questions** for every document.

Here, we have two contracts:

* 📄 **Short contract** — 12 pages
* 📚 **Long contract** — 216 pages

We will ask the same **3 questions** about both contracts.

This creates:

**2 contracts × 3 questions = 6 independent requests**

Instead of sending each request separately, we can submit all 6 requests together using an **Anthropic Message Batch**.

### Workflow

```text
12-page contract  ──┐
                    ├── 3 questions each ──→ 6 requests
216-page contract ──┘                           │
                                               ↓
                                          Message Batch
                                               ↓
                                         Check Status
                                               ↓
                                        Retrieve Results
```

Batching is useful for **evals**, where we often need to run the same test cases across many documents or prompts.

In this example, we can also compare how the model answers the same questions when working with a **short vs. long contract**.

#### Step 1 — Prepare the two contracts + 3 common questions

In [43]:
short_contract = "\n".join(page["text"] for page in short_document)
long_contract = "\n".join(page["text"] for page in long_document)

questions = [
    "What is the total contract value?",
    "What are the payment terms?",
    "What are the termination conditions?"
]

#### Step 2 — Create 6 requests and submit them as one batch

In [44]:
requests = []

for name, contract in [
    ("12_page_contract", short_contract),
    ("216_page_contract", long_contract)
]:
    for i, question in enumerate(questions, 1):
        requests.append({
            "custom_id": f"{name}_q{i}",
            "params": {
                "model": "claude-sonnet-4-6",
                "max_tokens": 500,
                "messages": [{
                    "role": "user",
                    "content": f"""
Analyze this contract and answer the question.

Contract:
{contract}

Question: {question}

Answer only using information from the contract.
"""
                }]
            }
        })

# 🚀 SEND ALL 6 REQUESTS TO ANTHROPIC AS ONE BATCH
batch = client.messages.batches.create(requests=requests)

show("5️⃣ Batch Submitted", f"Submitted {len(requests)} requests\nBatch ID: {batch.id}", color="cyan")

╭─ 5️⃣ Batch Submitted ─────────────────────────────╮                                                                
│                                                 │                                                                
│   Submitted 6 requests                          │                                                                
│   Batch ID: msgbatch_015uSxML3E5TFtQsA6cusv1u   │                                                                
│                                                 │                                                                
╰─────────────────────────────────────────────────╯                                                                



#### Step 3 — Wait for completion + retrieve all answers

⚠️ Note: Batch processing is asynchronous and may take some time to complete.

In [45]:
import time

while True:
    status = client.messages.batches.retrieve(batch.id)

    print(f"Batch status: {status.processing_status}")

    if status.processing_status == "ended":
        break

    time.sleep(2)

print("\n✅ Batch processing complete!")

for result in client.messages.batches.results(batch.id):
    if result.result.type == "succeeded":
        show(f"5️⃣ Batch Result — {result.custom_id}", result.result.message.content[0].text, color="green")

Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status: in_progress
Batch status

╭─ 5️⃣ Batch Result — 216_page_contract_q3 ─────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   Based on the contract, there are several termination conditions:                                              │
│                                                                                                                 │
│   ## Automatic Termination of Commitments                                                                       │
│   - **Closing Date Term Loan Commitments** terminate automatically immediately after the Term Loans are made    │
│   on the Closing Date                                                                                           │
│   - **Additional Term B Loan Commitments** terminate upon the making of the Additional Term B Loans on the      │
│   Fourth Amendment Effective Date                                   

╭─ 5️⃣ Batch Result — 216_page_contract_q2 ─────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   Based on the contract, here are the key payment terms:                                                        │
│                                                                                                                 │
│   ## Interest Rates                                                                                             │
│   - **SOFR Rate Loans**: Adjusted Term SOFR plus the Applicable Margin                                          │
│   - **Base Rate Loans**: Base Rate plus the Applicable Margin                                                   │
│   - **Applicable Margin**: Ranges from **7.25%-8.00%** for SOFR Rate Loans and **6.25%-7.00%** for Base Rate    │
│   Loans, based on Total Net Leverage Ratio                          

╭─ 5️⃣ Batch Result — 12_page_contract_q2 ──────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   ## Payment Terms                                                                                              │
│                                                                                                                 │
│   Based on the contract, the following payment terms apply:                                                     │
│                                                                                                                 │
│   ### General Payment Terms (Section 3.1)                                                                       │
│   - Fees and charges are **calculated and billed monthly**                                                      │
│   - AWS may bill **more frequently** if the account is suspected of 

╭─ 5️⃣ Batch Result — 216_page_contract_q1 ─────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   Based on the contract, the Additional Term B Loans requested under this Fourth Amendment have an aggregate    │
│   principal amount of **$170,000,000**.                                                                         │
│                                                                                                                 │
│   However, this amendment is to an existing Term Loan Credit Agreement that was originally dated March 4,       │
│   2022, with an initial term loan facility of **$450,000,000** on the Closing Date, plus subsequent             │
│   additional term loans and delayed draw term loans added through prior amendments. The contract does not       │
│   provide a single consolidated total of all outstanding term loans 

╭─ 5️⃣ Batch Result — 12_page_contract_q1 ──────────────────────────────────────────────────────╮                    
│                                                                                             │                    
│   Based on the contract, specifically in **Section 3.2(b)**, the total contract value is:   │                    
│                                                                                             │                    
│   **USD 35,000/- paid annually before the start of work.**                                  │                    
│                                                                                             │                    
╰─────────────────────────────────────────────────────────────────────────────────────────────╯                    



╭─ 5️⃣ Batch Result — 12_page_contract_q3 ──────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│    ## Termination Conditions                                                                                    │
│                                                                                                                 │
│    Based on the contract (Section 5.2), the following termination conditions apply:                             │
│                                                                                                                 │
│    ### Termination for Convenience                                                                              │
│    - **You (the customer)** may terminate the Agreement for any reason by providing notice and closing your     │
│    account for all Services                                         

### Why Use Message Batches?

You might be wondering:

> **"Couldn't we just use asynchronous calls to send all 6 requests at once?"**

Yes! We could.

The difference is **what we're optimizing for**.

With asynchronous calls, our application starts multiple requests concurrently and waits for their responses. This is useful when we need the results relatively quickly.

With a **Message Batch**, we submit a collection of independent requests as a background job. We don't need an immediate response. We can check the batch status later and retrieve the results when processing is complete.

For our contract example:

**2 contracts × 3 questions = 6 independent requests**

Instead of managing those six requests individually, we submit them as **one batch**.

### When should you use each?

* **Synchronous call** → "I need this answer now."
* **Async calls** → "I need several answers concurrently."
* **Message Batch** → "I have many independent jobs and can process them in the background."

## Chapter 6 — The Claude Agent SDK vs. the Messages API

Here's where the story changes gears.

Everything we've done so far — sync, streaming, async, batching — is still the same basic pattern: **we** build the prompt, **we** call `client.messages.create(...)`, and Claude sends back a single "completion." That's the **Messages API**: a low-level, request-in / response-out interface, exactly like a text-completion call.

The **Claude Agent SDK** is a different way of working with Claude. Instead of a single request/response, you hand Claude a task and it can:

* run in a loop across multiple turns instead of just one,
* use tools (files, bash, search, or your own custom tools) as part of solving the task,
* stream back a *sequence* of typed messages (system info, assistant turns, tool calls, a final result) instead of one flat object.

| | Messages API (`client.messages.create`) | Claude Agent SDK (`query`) |
|---|---|---|
| **Shape** | One request → one response ("completion") | One task → a stream of messages, possibly across multiple turns |
| **Tool use** | You manage tool calls yourself | Built-in tool loop (files, bash, custom tools, …) |
| **Best for** | Direct, single-shot or few-shot text generation — exactly what we've done in this notebook so far | Agentic workflows: multi-step research, coding, or document tasks where Claude should decide what to do next |

To make this concrete, we'll run **the exact same contract-analysis task** through both interfaces, side by side, so you can see the difference in code — and in what comes back.

### Installing the Claude Agent SDK

The Claude Agent SDK is built on top of the Claude Code CLI, so we need two things:

1. **Node.js** (already available in Colab) to run the Claude Code CLI.
2. The `claude-agent-sdk` Python package, plus the Claude Code CLI itself.

The SDK reads your API key from the `ANTHROPIC_API_KEY` environment variable, so we'll set that too.

In [47]:
!npm install -g @anthropic-ai/claude-code
!pip install -q -U claude-agent-sdk

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 2 packages in 6s
⠹npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.4 MB/s eta 0:00:00


In [46]:
import os

# Reuse the same key you already gave the `client` object above —
# the Agent SDK's CLI subprocess can't see `client`, so it needs
# the key via an environment variable instead.
os.environ["ANTHROPIC_API_KEY"] = client.api_key

### 🥊 Side-by-side: Messages API vs. Claude Agent SDK

We'll ask for the exact same thing from the exact same contract, using both approaches:

> Analyze this contract and identify the contract parties, effective date, contract value, and termination terms.

**A) Messages API** — the pattern we've used all notebook long: one call, one response.

In [48]:
task_prompt = f"""
Analyze this contract and identify:

1. Contract parties
2. Effective date
3. Contract value
4. Termination terms

Contract:
{contract_text}
"""

# A) Messages API — a single request/response, like a completion
messages_api_response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=[
        {"role": "user", "content": task_prompt}
    ]
)

show("📨 Messages API — Response", messages_api_response.content[0].text, color="cyan")

╭─ 📨 Messages API — Response ────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   # AWS Customer Agreement Analysis                                                                             │
│                                                                                                                 │
│   ## 1. Contract Parties                                                                                        │
│                                                                                                                 │
│   | Party | Details |                                                                                           │
│   |-------|---------|                                                                                           │
│   | **Service Provider** | Amazon Web Services (AWS) / applicable AWS C

**B) Claude Agent SDK** — the same task, but through `query()`, which is asynchronous and yields a *stream* of typed messages instead of a single object.

Note the differences in the code itself:

* We use `async for message in query(...)` instead of a single `client.messages.create(...)` call.
* Claude's actual answer arrives inside an `AssistantMessage`, as one or more `TextBlock`s — we have to look for it in the stream.
* Nothing here uses tools, but this is the same loop the SDK would use if Claude decided to read a file, run a search, or call a custom tool mid-task.

In [49]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock

async def ask_agent_sdk(prompt):
    options = ClaudeAgentOptions(
        model="claude-sonnet-4-6",
        max_turns=1,          # keep this a single-turn task, for a fair comparison
    )

    final_text = ""

    # B) Claude Agent SDK — a stream of messages, not one flat response
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    final_text += block.text

    return final_text


agent_sdk_response = await ask_agent_sdk(task_prompt)

show("🤖 Claude Agent SDK — Response", agent_sdk_response, color="magenta")

╭─ 🤖 Claude Agent SDK — Response ────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│   # AWS Customer Agreement — Contract Analysis                                                                  │
│                                                                                                                 │
│   ---                                                                                                           │
│                                                                                                                 │
│   ## 1. 🤝 Contract Parties                                                                                     │
│                                                                                                                 │
│   | Role | Party |                                                      

### 🔍 What did we just see?

Both calls answered the same question about the same contract — but notice what was different:

* **Messages API**: we got back one object, and the answer was sitting right there at `response.content[0].text`.
* **Claude Agent SDK**: we had to iterate over a *stream* of messages and pick out the `AssistantMessage` / `TextBlock` pieces — because the SDK is built to support multi-turn, tool-using agents, not just single completions.

For a simple, one-shot extraction task like this one, the Messages API is simpler and perfectly sufficient — which is why we used it for the rest of this notebook.

Where the **Claude Agent SDK** earns its keep is when the task isn't a single completion anymore: e.g. "go through this folder of contracts, flag anything with a non-standard termination clause, and write a summary file" — a multi-step job where Claude needs to decide what to look at next, not just answer one prompt.

### 🧭 Choosing between them

* **Messages API** → "I have one prompt and I want one answer." (Everything we did in Chapters 2–5.)
* **Claude Agent SDK** → "I have a task that may need multiple steps and tools to complete."

## 🏁 The Story So Far

Let's recap the journey:

1. We started with a contract PDF and watched it get broken into **tokens**, and saw the **context window** limit that comes with it.
2. We saw how **temperature** introduces randomness, leading to **non-determinism** — and why LLM testing needs to check for *correctness*, not exact-match text.
3. We picked the right **model** for the job and used **adaptive thinking** to let Claude decide how much reasoning a task deserves.
4. We wrote **zero-shot, one-shot, and few-shot** prompts and watched the output improve with the right amount of guidance.
5. We called Claude through the **Messages API** four different ways: synchronous, streaming, async, and batched.
6. Finally, we placed the **Messages API** side by side with the **Claude Agent SDK** and saw exactly how a single completion differs from an agentic, multi-message workflow.

You now have a working mental model for both interfaces — and, just as importantly, for *when* to reach for each one in a real legal-tech application.